In [17]:
import pandas as pd
import litellm
import time
import re
import os
import sys

# =========================
# CONFIG
# =========================
MODEL_NAME = "gpt-4o"
BENCHMARK_FILE = "100_HU_Riddles_Benchmark_Questions.tsv"

# Run a slice for testing; change to 1–100 later
ID_MIN = 1
ID_MAX = 100

OUTPUT_FILE = "gpt4o_open_set_ids45_55_answer_reasoning.tsv"

SLEEP_BETWEEN_CALLS = 1.0
MAX_ANSWER_TRIES = 3

# Console output (file will be CLEAN regardless)
SHOW_EXPECTED_ON_SCREEN = True     # not written to file
SHOW_RIDDLE_ON_SCREEN = False      # not written to file

# =========================
# SAFE FEW-SHOT (NO benchmark items, NO expected answers, no derived lists)
# =========================
FEW_SHOT_HUNGARIAN = """
# Generic Hungarian anchors (not benchmark items)
Szilvásgombóc | Hungarian plum dumpling. [Szilvásgombóc]
Rétes | Hungarian strudel. [Rétes]
Gundel palacsinta | Famous Hungarian dessert. [Gundel palacsinta]
Béla Bartók | Hungarian composer. [Béla Bartók]
Tihany | Village at Lake Balaton. [Tihany]
---
"""

# =========================
# PROMPTS
# =========================
SYSTEM_PROMPT_ANSWER = (
    "You are an expert Hungarian cultural recall assistant. "
    "Return ONE canonical Hungarian term (1–4 words). "
    "No foreign names, no international analogues, no descriptive paraphrases. "
    "Output only the final answer in square brackets: [Válasz]."
)

SYSTEM_PROMPT_CHECK = (
    "Ön egy szigorú ellenőr. "
    "A feladat: eldönteni, hogy a javasolt rövid válasz közvetlenül illik-e a rejtvényre. "
    "Ha a válasz túl általános, nem magyar kulturális kifejezés, vagy nem következik a rejtvényből, utasítsa el. "
    "Kimenet kizárólag:\n"
    "OK\n"
    "vagy\n"
    "RETRY: <max 12 szó>\n"
)

SYSTEM_PROMPT_REASONING = (
    "Ön egy segítőkész, precíz magyar asszisztens. "
    "KIZÁRÓLAG magyarul válaszoljon. "
    "Az indoklás 1–2 mondat legyen, tömör és tényszerű. "
    "A MEGADOTT választ magyarázza, és ne módosítsa."
)

# =========================
# HELPERS
# =========================
def run_gpt(system_prompt: str, user_prompt: str, temp: float = 0.0, max_tokens: int = 256) -> str:
    params = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temp,
        "max_tokens": max_tokens,
        "api_key": os.environ.get("OPENAI_API_KEY"),
        "tools": [],
        "tool_choice": "none",
    }
    for attempt in range(3):
        try:
            resp = litellm.completion(**params)
            return (resp.choices[0].message.content or "").strip()
        except Exception as e:
            if attempt < 2:
                time.sleep(2 * (attempt + 1))
            else:
                return f"API_ERROR: {str(e)[:200]}"
    return "API_ERROR: Max retries exhausted"

def extract_bracket(text: str) -> str:
    if not text:
        return ""
    m = re.findall(r"\[(.*?)\]", text)
    return m[-1].strip() if m else text.strip()

def normalize_answer(ans: str) -> str:
    a = (ans or "").strip()
    a = re.sub(r"^\s*Válasz\s*:\s*", "", a, flags=re.IGNORECASE)
    a = re.sub(r"^\s*Answer\s*:\s*", "", a, flags=re.IGNORECASE)
    return a.strip()

def looks_invalid_short_answer(ans: str) -> bool:
    if not ans:
        return True
    s = ans.strip()
    if len(s.split()) > 4:
        return True
    low = s.lower()
    if low in {"válasz", "ismeretlen", "indoklás"}:
        return True
    if any(p in s for p in [".", "!", "?", ";", ":"]):
        return True
    if any(x in f" {low} " for x in [" az ", " egy ", " ami ", " amely ", " mert ", " vagyis "]):
        return True
    return False

def self_check_fit(riddle_text: str, proposed: str) -> bool:
    prompt = (
        "Csak a rejtvény és a válasz alapján dönts.\n"
        "Kimenet: OK vagy RETRY: ...\n\n"
        f"Rejtvény: {riddle_text}\n"
        f"Javasolt válasz: {proposed}\n"
    )
    out = run_gpt(SYSTEM_PROMPT_CHECK, prompt, temp=0.0, max_tokens=80).strip()
    return out.startswith("OK")

def enforce_hu_only(text: str) -> str:
    if not text:
        return "Nincs indoklás."
    low = text.lower()
    eng_markers = [" is ", " are ", " the ", " because ", " refers to ", " known ", " famous "]
    if any(m in low for m in eng_markers):
        repaired = run_gpt(
            SYSTEM_PROMPT_REASONING,
            "Írd át kizárólag magyarra, 1–2 mondatban, tömören:\n\n" + text,
            temp=0.0,
            max_tokens=140
        ).strip()
        return repaired
    return text

# =========================
# MAIN
# =========================
if __name__ == "__main__":
    print("🔑 OpenAI API Key:", "✓ Set" if os.environ.get("OPENAI_API_KEY") else "✗ MISSING")
    if not os.environ.get("OPENAI_API_KEY"):
        print("🛑 Missing OPENAI_API_KEY in environment.")
        sys.exit(1)

    try:
        df_all = pd.read_csv(BENCHMARK_FILE, sep="\t")
    except Exception as e:
        print(f"🛑 Failed to load TSV '{BENCHMARK_FILE}': {e}")
        sys.exit(1)

    df_all["ID_num"] = pd.to_numeric(df_all["ID"], errors="coerce")
    df = df_all[(df_all["ID_num"] >= ID_MIN) & (df_all["ID_num"] <= ID_MAX)].copy()
    df = df.sort_values("ID_num")

    if df.empty:
        print(f"🛑 No rows found for ID {ID_MIN}–{ID_MAX}.")
        sys.exit(1)

    print(f"\n=== OPEN-SET RUN (model never sees expected answers) IDs {ID_MIN}–{ID_MAX} ({len(df)} riddles) ===\n")

    out_rows = []

    for _, r in df.iterrows():
        rid = int(r["ID_num"])
        topic = str(r.get("topic", "")).strip()
        riddle_text = str(r.get("riddle_text", "")).strip()

        # Expected is used ONLY for optional screen reporting (never sent to model; never saved)
        expected = str(r.get("reference_answer", "")).strip()

        if SHOW_EXPECTED_ON_SCREEN:
            print(f"[ID {rid}] Expected: {expected}")
        else:
            print(f"[ID {rid}]")

        if SHOW_RIDDLE_ON_SCREEN:
            print(f"Riddle: {riddle_text}")

        best_ans = ""

        # Step A: short answer with retry + self-check (open-set)
        for _ in range(MAX_ANSWER_TRIES):
            ans_prompt = (
                FEW_SHOT_HUNGARIAN
                + f"\n[Téma: {topic}]\n"
                + f"[Rejtvény: {riddle_text}]\n"
                + "Adj 1 magyar kulturális kifejezést szögletes zárójelben."
            )
            raw = run_gpt(SYSTEM_PROMPT_ANSWER, ans_prompt, temp=0.0, max_tokens=80)
            cand = normalize_answer(extract_bracket(raw))

            if looks_invalid_short_answer(cand):
                best_ans = cand
                continue

            if self_check_fit(riddle_text, cand):
                best_ans = cand
                break

            best_ans = cand
            time.sleep(SLEEP_BETWEEN_CALLS)

        time.sleep(SLEEP_BETWEEN_CALLS)

        # Step B: Hungarian-only reasoning for the GIVEN answer
        reasoning_prompt = (
            "Kizárólag magyarul. 1–2 mondat, tömören.\n"
            "A feladat: magyarázd meg, miért illik az ADOTT válasz a rejtvényre.\n\n"
            f"Rejtvény: {riddle_text}\n"
            f"Adott válasz: {best_ans}\n"
            "Indoklás:"
        )
        reasoning = run_gpt(SYSTEM_PROMPT_REASONING, reasoning_prompt, temp=0.0, max_tokens=160).strip()
        reasoning = enforce_hu_only(reasoning)

        # Clean screen output
        print(f" -> [{best_ans}] | {reasoning}\n")

        # CLEAN FILE OUTPUT (no Expected, no Debug, no internal notes)
        out_rows.append({
            "ID": rid,
            "Topic": topic,
            "Riddle": riddle_text,
            "Final_Answer": best_ans,
            "Reasoning": reasoning
        })

        time.sleep(SLEEP_BETWEEN_CALLS)

    out_df = pd.DataFrame(out_rows)
    out_df.to_csv(OUTPUT_FILE, index=False, sep="\t")
    print(f"✅ Saved: {OUTPUT_FILE}")


🔑 OpenAI API Key: ✓ Set

=== OPEN-SET RUN (model never sees expected answers) IDs 1–100 (100 riddles) ===

[ID 1] Expected: Unicum
 -> [Unicum] | Az "Unicum" egy gömb alakú üvegben forgalmazott keserű likőr, amelyet gyakran emlegetnek gyomorkeserűként, így nem édes a hasnak. A vöröskeresztes utalás a likőr címkéjén található vörös kereszt szimbólumra vonatkozik.

[ID 2] Expected: Csaba és Gyula
 -> [Válasz] | A "Válasz" szó egyaránt lehet keresztnév és városnév, így a rejtvényben a kisfiú nem tudja, hogy a saját nevét vagy a célállomást kérdezik tőle. Ez a kettősség adja a vicc alapját.

[ID 3] Expected: Kürtőskalács
 -> [Kürtőskalács] | A kürtőskalácsot hagyományosan kémény alakú formára tekerik, és parázs felett sütik, így a "kémény belsejében készítik" kifejezés erre utal. Nem hangszerre hasonlít, mivel a kürtőskalács nem egy hangszer, hanem egy édesség.

[ID 4] Expected: Szaloncukor
 -> [Szalonka] | A "szalonka" egy madár, amelynek neve a "szalon" szóból származik, de valójában fák